# Tech Challenge Fase 3 - Engenharia de Features
## Criacao de Features Derivadas para Previsao de Atrasos

**Disciplina**: Machine Learning Engineering - Fase 3  
**Instituicao**: FIAP PosTech - AI Engineering & Machine Learning

---

## Indice
1. [Configuracao do Ambiente](#1-configuracao)
2. [Carregamento dos Dados](#2-carregamento)
3. [Criacao de Variaveis Target](#3-target)
4. [Features Temporais](#4-temporal)
5. [Features Geograficas](#5-geograficas)
6. [Features Agregadas](#6-agregadas)
7. [Processamento Completo](#7-completo)
8. [Salvamento dos Dados Processados](#8-salvamento)

---

## 1. Configuracao do Ambiente <a id='1-configuracao'></a>

### 1.1 Importacao de Bibliotecas

In [1]:
# Manipulacao de dados
import pandas as pd
import numpy as np
import sys
import os

# Adicionar path do projeto
sys.path.append('../src')

# Importar modulos do projeto
from preprocessing.data_loader import load_all_datasets
from preprocessing.feature_engineering import (
    create_target_variables,
    create_temporal_features,
    create_geographic_features,
    create_aggregated_features,
    create_all_features,
    get_feature_names
)

# Configuracoes
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Bibliotecas importadas com sucesso!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

Bibliotecas importadas com sucesso!
Pandas: 2.1.4
NumPy: 1.26.2


### 1.2 Definicao de Constantes

In [2]:
# Caminhos dos arquivos
DATA_PATH = '../data/raw/'
FLIGHTS_FILE = f'{DATA_PATH}flights.csv'
AIRPORTS_FILE = f'{DATA_PATH}airports.csv'
AIRLINES_FILE = f'{DATA_PATH}airlines.csv'

# Caminho para salvar dados processados
PROCESSED_DATA_PATH = '../data/processed/'

# Constantes de negocio
DELAY_THRESHOLD = 15  # Voo atrasado se atraso > 15 minutos
RANDOM_STATE = 42

# Usar dataset completo
USE_FULL_DATASET = True  # True = todos os 5.819.080 registros
SAMPLE_SIZE = None if USE_FULL_DATASET else 100000

print(f"Configuracoes:")
print(f"  - Threshold de atraso: {DELAY_THRESHOLD} minutos")
print(f"  - Dataset: {'COMPLETO (5.819.080 registros)' if USE_FULL_DATASET else f'AMOSTRA ({SAMPLE_SIZE:,} registros)'}")

Configuracoes:
  - Threshold de atraso: 15 minutos
  - Dataset: COMPLETO (5.819.080 registros)


---
## 2. Carregamento dos Dados <a id='2-carregamento'></a>

In [3]:
print("Carregando datasets...")
print("Isso pode levar alguns minutos para o dataset completo...\n")

df_flights, df_airports, df_airlines = load_all_datasets(
    FLIGHTS_FILE,
    AIRPORTS_FILE,
    AIRLINES_FILE,
    flights_nrows=SAMPLE_SIZE,
    optimize_memory=True,
    verbose=True
)

print(f"\nDados carregados:")
print(f"  - Voos: {len(df_flights):,} registros")
print(f"  - Aeroportos: {len(df_airports)} registros")
print(f"  - Companhias: {len(df_airlines)} registros")

Carregando datasets...
Isso pode levar alguns minutos para o dataset completo...


CARREGANDO TODOS OS DATASETS

CARREGANDO DATASET DE VOOS
Modo: Dataset completo

Dataset carregado com sucesso!
Dimensões: 5,819,079 linhas x 31 colunas
Memória utilizada: 505.53 MB

Carregando dataset de aeroportos...
322 aeroportos carregados!
Memória: 120.03 KB

Carregando dataset de companhias aéreas...
14 companhias aéreas carregadas!
Memória: 2.55 KB

TODOS OS DATASETS CARREGADOS COM SUCESSO!
Memória total utilizada: 505.65 MB


Dados carregados:
  - Voos: 5,819,079 registros
  - Aeroportos: 322 registros
  - Companhias: 14 registros


---
## 3. Criacao de Variaveis Target <a id='3-target'></a>

Vamos criar as variaveis target para nossos modelos:
- **IS_DELAYED**: Classificacao binaria (0 ou 1)
- **DELAY_MINUTES**: Regressao (valor continuo)

In [4]:
df_flights = create_target_variables(
    df_flights,
    delay_threshold=DELAY_THRESHOLD,
    verbose=True
)

VARIAVEIS TARGET CRIADAS

1. IS_DELAYED (Classificacao Binaria)
   - Threshold: 15 minutos
   - Voos atrasados: 1,018,558 (17.50%)
   - Voos no horario: 4,800,521 (82.50%)

2. DELAY_MINUTES (Regressao)
   - Media: 9.37 minutos
   - Mediana: -2.00 minutos



In [5]:
# Visualizar primeiras linhas com as novas colunas
print("Primeiras 10 linhas com targets:")
display(df_flights[['DEPARTURE_DELAY', 'IS_DELAYED', 'DELAY_MINUTES']].head(10))

Primeiras 10 linhas com targets:


,DEPARTURE_DELAY,IS_DELAYED,DELAY_MINUTES
0,-11.00,0,-11.00
1,-8.00,0,-8.00
2,-2.00,0,-2.00
3,-5.00,0,-5.00
4,-1.00,0,-1.00
5,-5.00,0,-5.00
6,-6.00,0,-6.00
7,14.00,0,14.00
8,-11.00,0,-11.00
9,3.00,0,3.00


---
## 4. Features Temporais <a id='4-temporal'></a>

Features baseadas em data e horario:
- Periodo do dia
- Fim de semana
- Estacao do ano
- Horarios de pico

In [6]:
df_flights = create_temporal_features(
    df_flights,
    verbose=True
)

FEATURES TEMPORAIS CRIADAS

1. PERIOD_OF_DAY:
PERIOD_OF_DAY
Madrugada     140597
Manha        2263032
Noite        1274340
Tarde        2141110
Name: count, dtype: int64

2. IS_WEEKEND:
   - Fim de semana: 1,518,309 (26.09%)

3. SEASON:
SEASON
Verao        1535151
Primavera    1486456
Outono       1419083
Inverno      1378389
Name: count, dtype: int64

4. Horarios de Pico:
   - Manha (6-9h): 1,533,304 (26.35%)
   - Noite (17-20h): 1,315,512 (22.61%)



In [7]:
# Visualizar features temporais criadas
temporal_cols = ['HOUR', 'PERIOD_OF_DAY', 'IS_WEEKEND', 'SEASON', 'IS_MORNING_RUSH', 'IS_EVENING_RUSH']
print("Amostra de features temporais:")
display(df_flights[temporal_cols].head(10))

Amostra de features temporais:


,HOUR,PERIOD_OF_DAY,IS_WEEKEND,SEASON,IS_MORNING_RUSH,IS_EVENING_RUSH
0,0,Madrugada,0,Inverno,0,0
1,0,Madrugada,0,Inverno,0,0
2,0,Madrugada,0,Inverno,0,0
3,0,Madrugada,0,Inverno,0,0
4,0,Madrugada,0,Inverno,0,0
5,0,Madrugada,0,Inverno,0,0
6,0,Madrugada,0,Inverno,0,0
7,0,Madrugada,0,Inverno,0,0
8,0,Madrugada,0,Inverno,0,0
9,0,Madrugada,0,Inverno,0,0


In [8]:
# Analise: Taxa de atraso por periodo do dia
print("Taxa de atraso por periodo do dia:")
delay_by_period = df_flights['IS_DELAYED'].astype('float').groupby(df_flights['PERIOD_OF_DAY'], observed=True).mean().sort_values(ascending=False)
print(delay_by_period)

print("\nTaxa de atraso: Fim de semana vs Dia de semana:")
delay_by_weekend = df_flights['IS_DELAYED'].astype('float').groupby(df_flights['IS_WEEKEND'], observed=True).mean()
print(f"  Dia de semana: {delay_by_weekend[0]:.4f}")
print(f"  Fim de semana: {delay_by_weekend[1]:.4f}")

Taxa de atraso por periodo do dia:
PERIOD_OF_DAY
Noite       0.25
Tarde       0.20
Manha       0.11
Madrugada   0.07
Name: IS_DELAYED, dtype: float64

Taxa de atraso: Fim de semana vs Dia de semana:
  Dia de semana: 0.1780
  Fim de semana: 0.1665


---
## 5. Features Geograficas <a id='5-geograficas'></a>

Features baseadas em localizacao:
- Estados de origem e destino
- Voo domestico (mesmo estado)
- Coordenadas geograficas

In [9]:
df_flights = create_geographic_features(
    df_flights,
    df_airports,
    verbose=True
)

FEATURES GEOGRAFICAS CRIADAS

1. Estados de Origem:
   - Total de estados: 54

2. IS_DOMESTIC:
   - Voos domesticos (mesmo estado): 592,162 (10.18%)

3. Coordenadas geograficas adicionadas para origem e destino



In [10]:
# Visualizar features geograficas criadas
geo_cols = ['ORIGIN_STATE', 'DEST_STATE', 'IS_DOMESTIC', 'ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE']
print("Amostra de features geograficas:")
display(df_flights[geo_cols].head(10))

Amostra de features geograficas:


,ORIGIN_STATE,DEST_STATE,IS_DOMESTIC,ORIGIN_LATITUDE,ORIGIN_LONGITUDE
0,AK,WA,0,61.17,-150.00
1,CA,FL,0,33.94,-118.41
2,CA,NC,0,37.62,-122.37
3,CA,FL,0,33.94,-118.41
4,WA,AK,0,47.45,-122.31
5,CA,MN,0,37.62,-122.37
6,NV,MN,0,36.08,-115.15
7,CA,NC,0,33.94,-118.41
8,CA,TX,0,37.62,-122.37
9,NV,GA,0,36.08,-115.15


In [11]:
# Analise: Taxa de atraso por tipo de voo
print("Taxa de atraso: Domestico vs Interestadual:")
delay_by_domestic = df_flights['IS_DELAYED'].astype('float').groupby(df_flights['IS_DOMESTIC'], observed=True).mean()
print(f"  Interestadual (diferente estado): {delay_by_domestic[0]:.4f}")
print(f"  Domestico (mesmo estado): {delay_by_domestic[1]:.4f}")

print("\nTop 5 estados de origem com maior taxa de atraso:")
delay_by_state = df_flights.groupby('ORIGIN_STATE', observed=True)['IS_DELAYED'].agg(['mean', 'count'])
delay_by_state = delay_by_state[delay_by_state['count'] >= 100]  # Min 100 voos
print(delay_by_state.sort_values('mean', ascending=False).head())

Taxa de atraso: Domestico vs Interestadual:
  Interestadual (diferente estado): 0.1760
  Domestico (mesmo estado): 0.1661

Top 5 estados de origem com maior taxa de atraso:
              mean   count
ORIGIN_STATE              
DE            0.37     100
GU            0.26     334
MD            0.23   86079
IL            0.23  381644
NJ            0.22  108174


---
## 6. Features Agregadas <a id='6-agregadas'></a>

Features baseadas em taxas historicas de atraso:
- Por companhia aerea
- Por aeroporto de origem
- Por aeroporto de destino
- Por rota

**IMPORTANTE**: Em producao, estas features devem ser calculadas apenas no conjunto de treino!

In [12]:
df_flights = create_aggregated_features(
    df_flights,
    verbose=True
)

FEATURES AGREGADAS CRIADAS

1. AIRLINE_DELAY_RATE:
   - Media: 0.1750
   - Min: 0.0686
   - Max: 0.2610

2. ORIGIN_DELAY_RATE:
   - Media: 0.1750
   - Min: 0.0000
   - Max: 0.4444

3. ROUTE_DELAY_RATE:
   - Media: 0.1750
   - Rotas unicas: 8,609



In [13]:
# Visualizar features agregadas criadas
agg_cols = ['AIRLINE_DELAY_RATE', 'ORIGIN_DELAY_RATE', 'DEST_DELAY_RATE', 'ROUTE_DELAY_RATE']
print("Amostra de features agregadas:")
display(df_flights[agg_cols].head(10))

print("\nEstatisticas das features agregadas:")
display(df_flights[agg_cols].describe())

Amostra de features agregadas:


,AIRLINE_DELAY_RATE,ORIGIN_DELAY_RATE,DEST_DELAY_RATE,ROUTE_DELAY_RATE
0,0.10,0.12,0.15,0.09
1,0.16,0.20,0.21,0.12
2,0.14,0.20,0.13,0.09
3,0.16,0.20,0.17,0.14
4,0.10,0.15,0.16,0.15
5,0.13,0.20,0.15,0.14
6,0.26,0.21,0.15,0.15
7,0.14,0.20,0.13,0.16
8,0.16,0.20,0.17,0.19
9,0.13,0.21,0.14,0.14



Estatisticas das features agregadas:


,AIRLINE_DELAY_RATE,ORIGIN_DELAY_RATE,DEST_DELAY_RATE,ROUTE_DELAY_RATE
count,5819079.00,5819079.00,5819079.00,5819079.00
mean,0.18,0.18,0.18,0.18
std,0.04,0.04,0.03,0.06
min,0.07,0.00,0.00,0.00
25%,0.16,0.15,0.15,0.14
50%,0.16,0.17,0.18,0.18
75%,0.20,0.20,0.20,0.21
max,0.26,0.44,0.36,1.00


In [14]:
# Analise: Companhias com maiores taxas de atraso
print("Top 10 companhias com maior taxa de atraso:")
airline_delay = df_flights.groupby('AIRLINE', observed=True)['AIRLINE_DELAY_RATE'].first().sort_values(ascending=False)
print(airline_delay.head(10))

Top 10 companhias com maior taxa de atraso:
AIRLINE
NK   0.26
UA   0.23
F9   0.22
B6   0.21
WN   0.20
MQ   0.19
VX   0.17
AA   0.16
EV   0.16
OO   0.16
Name: AIRLINE_DELAY_RATE, dtype: float64


---
## 7. Processamento Completo <a id='7-completo'></a>

Agora vamos demonstrar como criar todas as features de uma vez usando a funcao `create_all_features()`.

In [15]:
# Recarregar dados para demonstrar o processo completo
print("Recarregando dados para demonstrar processo completo...\n")

df_flights_fresh, df_airports_fresh, df_airlines_fresh = load_all_datasets(
    FLIGHTS_FILE,
    AIRPORTS_FILE,
    AIRLINES_FILE,
    flights_nrows=SAMPLE_SIZE,
    optimize_memory=True,
    verbose=False
)

print(f"Dados recarregados: {len(df_flights_fresh):,} voos\n")

Recarregando dados para demonstrar processo completo...

Dados recarregados: 5,819,079 voos



In [16]:
# Criar todas as features de uma vez
df_enriched = create_all_features(
    df_flights_fresh,
    df_airports_fresh,
    delay_threshold=DELAY_THRESHOLD,
    verbose=True
)


INICIANDO ENGENHARIA DE FEATURES COMPLETA

VARIAVEIS TARGET CRIADAS

1. IS_DELAYED (Classificacao Binaria)
   - Threshold: 15 minutos
   - Voos atrasados: 1,018,558 (17.50%)
   - Voos no horario: 4,800,521 (82.50%)

2. DELAY_MINUTES (Regressao)
   - Media: 9.37 minutos
   - Mediana: -2.00 minutos

FEATURES TEMPORAIS CRIADAS

1. PERIOD_OF_DAY:
PERIOD_OF_DAY
Madrugada     140597
Manha        2263032
Noite        1274340
Tarde        2141110
Name: count, dtype: int64

2. IS_WEEKEND:
   - Fim de semana: 1,518,309 (26.09%)

3. SEASON:
SEASON
Verao        1535151
Primavera    1486456
Outono       1419083
Inverno      1378389
Name: count, dtype: int64

4. Horarios de Pico:
   - Manha (6-9h): 1,533,304 (26.35%)
   - Noite (17-20h): 1,315,512 (22.61%)

FEATURES GEOGRAFICAS CRIADAS

1. Estados de Origem:
   - Total de estados: 54

2. IS_DOMESTIC:
   - Voos domesticos (mesmo estado): 592,162 (10.18%)

3. Coordenadas geograficas adicionadas para origem e destino

FEATURES AGREGADAS CRIADAS

1. AI

In [17]:
# Visualizar todas as features criadas por categoria
features = get_feature_names(include_targets=True, include_original=True)

print("="*80)
print("FEATURES CRIADAS POR CATEGORIA")
print("="*80)

for category, feature_list in features.items():
    print(f"\n{category.upper()}:")
    for feat in feature_list:
        if feat in df_enriched.columns:
            print(f"  - {feat}")

FEATURES CRIADAS POR CATEGORIA

TARGET:
  - IS_DELAYED
  - DELAY_MINUTES

TEMPORAL:
  - HOUR
  - PERIOD_OF_DAY
  - IS_WEEKEND
  - SEASON
  - IS_MORNING_RUSH
  - IS_EVENING_RUSH

GEOGRAPHIC:
  - ORIGIN_STATE
  - DEST_STATE
  - IS_DOMESTIC
  - ORIGIN_LATITUDE
  - ORIGIN_LONGITUDE
  - DEST_LATITUDE
  - DEST_LONGITUDE

AGGREGATED:
  - AIRLINE_DELAY_RATE
  - ORIGIN_DELAY_RATE
  - DEST_DELAY_RATE
  - ROUTE_DELAY_RATE

ORIGINAL:
  - YEAR
  - MONTH
  - DAY
  - DAY_OF_WEEK
  - AIRLINE
  - ORIGIN_AIRPORT
  - DESTINATION_AIRPORT
  - SCHEDULED_DEPARTURE
  - SCHEDULED_TIME
  - DISTANCE


In [18]:
# Resumo do dataset enriquecido
print("="*80)
print("RESUMO DO DATASET ENRIQUECIDO")
print("="*80)
print(f"\nDimensoes: {df_enriched.shape[0]:,} linhas x {df_enriched.shape[1]} colunas")
print(f"Memoria: {df_enriched.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nColunas originais: {len(df_flights_fresh.columns)}")
print(f"Colunas adicionadas: {len(df_enriched.columns) - len(df_flights_fresh.columns)}")
print(f"Total de colunas: {len(df_enriched.columns)}")

RESUMO DO DATASET ENRIQUECIDO

Dimensoes: 5,819,079 linhas x 51 colunas
Memoria: 2179.55 MB

Colunas originais: 31
Colunas adicionadas: 20
Total de colunas: 51


---
## 8. Salvamento dos Dados Processados <a id='8-salvamento'></a>

Vamos salvar o dataset enriquecido para uso posterior na modelagem.

In [19]:
# Criar pasta de dados processados se nao existir
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Nome do arquivo
output_file = f"{PROCESSED_DATA_PATH}flights_enriched.csv"

print(f"Salvando dataset enriquecido em: {output_file}")
print("Isso pode levar alguns minutos...\n")

# Salvar
df_enriched.to_csv(output_file, index=False)

# Verificar tamanho do arquivo
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"Dataset salvo com sucesso!")
print(f"Tamanho do arquivo: {file_size_mb:.2f} MB")

Salvando dataset enriquecido em: ../data/processed/flights_enriched.csv
Isso pode levar alguns minutos...

Dataset salvo com sucesso!
Tamanho do arquivo: 1591.84 MB


In [20]:
# Salvar tambem uma versao otimizada em parquet (mais rapido para leitura)
output_parquet = f"{PROCESSED_DATA_PATH}flights_enriched.parquet"

print(f"\nSalvando tambem em formato Parquet: {output_parquet}")
df_enriched.to_parquet(output_parquet, index=False, compression='snappy')

# Verificar tamanho
parquet_size_mb = os.path.getsize(output_parquet) / (1024 * 1024)
print(f"Arquivo Parquet salvo!")
print(f"Tamanho: {parquet_size_mb:.2f} MB (reducao de {((file_size_mb - parquet_size_mb) / file_size_mb * 100):.1f}%)")


Salvando tambem em formato Parquet: ../data/processed/flights_enriched.parquet
Arquivo Parquet salvo!
Tamanho: 204.74 MB (reducao de 87.1%)


### 8.1 Verificacao dos Dados Salvos

In [21]:
# Testar carregamento do arquivo salvo
print("Testando carregamento do arquivo salvo...")
df_test = pd.read_parquet(output_parquet)

print(f"\nArquivo carregado com sucesso!")
print(f"Dimensoes: {df_test.shape[0]:,} x {df_test.shape[1]}")
print(f"\nPrimeiras 5 linhas:")
display(df_test.head())

Testando carregamento do arquivo salvo...

Arquivo carregado com sucesso!
Dimensoes: 5,819,079 x 51

Primeiras 5 linhas:


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,IS_DELAYED,DELAY_MINUTES,HOUR,PERIOD_OF_DAY,IS_WEEKEND,SEASON,IS_MORNING_RUSH,IS_EVENING_RUSH,ORIGIN_STATE,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,DEST_STATE,DEST_LATITUDE,DEST_LONGITUDE,IS_DOMESTIC,AIRLINE_DELAY_RATE,ORIGIN_DELAY_RATE,DEST_DELAY_RATE,ROUTE,ROUTE_DELAY_RATE
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5.00,2354.00,-11.00,21.00,15.00,205.00,194.00,169.00,1448,404.00,4.00,430.00,408.00,-22.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-11.00,0,Madrugada,0,Inverno,0,0,AK,61.17,-150.00,WA,47.45,-122.31,0,0.10,0.12,0.15,ANC-SEA,0.09
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10.00,2.00,-8.00,12.00,14.00,280.00,279.00,263.00,2330,737.00,4.00,750.00,741.00,-9.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-8.00,0,Madrugada,0,Inverno,0,0,CA,33.94,-118.41,FL,26.68,-80.10,0,0.16,0.20,0.21,LAX-PBI,0.12
2,2015,1,1,4,US,840,N171US,SFO,CLT,20.00,18.00,-2.00,16.00,34.00,286.00,293.00,266.00,2296,800.00,11.00,806.00,811.00,5.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-2.00,0,Madrugada,0,Inverno,0,0,CA,37.62,-122.37,NC,35.21,-80.94,0,0.14,0.20,0.13,SFO-CLT,0.09
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20.00,15.00,-5.00,15.00,30.00,285.00,281.00,258.00,2342,748.00,8.00,805.00,756.00,-9.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-5.00,0,Madrugada,0,Inverno,0,0,CA,33.94,-118.41,FL,25.79,-80.29,0,0.16,0.20,0.17,LAX-MIA,0.14
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25.00,24.00,-1.00,11.00,35.00,235.00,215.00,199.00,1448,254.00,5.00,320.00,259.00,-21.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-1.00,0,Madrugada,0,Inverno,0,0,WA,47.45,-122.31,AK,61.17,-150.00,0,0.10,0.15,0.16,SEA-ANC,0.15


---

## Conclusao

Neste notebook, criamos features derivadas para o modelo de previsao de atrasos:

**Features Criadas**:
- **Target**: IS_DELAYED, DELAY_MINUTES
- **Temporais**: HOUR, PERIOD_OF_DAY, IS_WEEKEND, SEASON, horarios de pico
- **Geograficas**: Estados, coordenadas, voo domestico
- **Agregadas**: Taxas historicas por companhia, aeroporto e rota

**Proximos Passos**:
1. Tratamento de valores ausentes
2. Preparacao para modelagem (train/test split, encoding)
3. Modelagem supervisionada (classificacao e regressao)
4. Modelagem nao supervisionada (clustering, PCA)